# Hierarchy-only and late-fusion models at 200%

Train the hierarchy-only and hierarchical + high-level late-fusion models on
the full **200% cohort**: official train, validation, and test members from
archives 1–8, plus all unique exemplars from archives 1–8. Both models use
the same generated manifest. Results are also reported on the old archive-1–4
test/exemplar slice for direct comparison with historical experiments.

The notebook is resumable within and across Kaggle sessions. Each run has an
independent time limit, optimizer backup, best checkpoint, result directory,
and provenance signature. The shared result archive is refreshed after every
run. A timeout evaluates the best checkpoint seen so far; a later session then
continues from the periodic optimizer backup.

The fusion run also needs `wa_high_level_archives1_8.pt`. Attach that small cache
as a Kaggle input and set `HIGH_LEVEL_CACHE_SOURCE`, or leave it as `None` to
auto-discover a uniquely named attached file. It is deliberately not embedded
in this repository or duplicated into the much larger tensor snapshot. Build
it locally with `python scripts/build_high_level_cache.py`, then upload only
the resulting cache file to Kaggle.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
import torch

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "ll-hls4ml"
RESULTS_DIR = WORK / "results"
CONFIG_DIR = WORK / "configs"

REPO_URL = "https://github.com/brios-polimi/ll-hls4ml.git"
# Update this after committing the notebook and sparse-relation DDP fix.
REPO_REF = "bdfddbb16c454582e87eda1accbb7c12d152a4b3"
TENSOR_REPO_ID = "BrendanRios/wa-hls4ml-tensors-hierarchical"
TENSOR_REVISION = "32bc0272f1a1f53fbf051371ca7e2728bba60c77"
HF_CACHE_DIR = Path("/tmp/wa_hls4ml_hierarchy_hf_cache")
TENSOR_DOWNLOAD_DIR = WORK / "hierarchy_tensors"
DOWNLOAD_HEARTBEAT_SECONDS = 60
DOWNLOAD_STALL_WARNING_MINUTES = 5

# huggingface_hub reads these when it is imported in the download cell.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_XET_CACHE"] = str(HF_CACHE_DIR / "xet")

# Set this to the cache file in an attached Kaggle input. When None, the
# notebook searches /kaggle/input for one unambiguous file with this name.
HIGH_LEVEL_CACHE_SOURCE = None

# Attach the zip produced by an earlier session and point this at its Kaggle
# input directory. Leave None for the first session.
PREVIOUS_RESULTS_ROOT = None

# Keep True after the sparse-relation DDP fix is in REPO_REF. Set False only
# as a one-GPU fallback when testing an older repository revision.
USE_DDP = True
ACTIVE_RUNS = ["hierarchical", "fusion"]
TRAIN_BUDGETS = {
    "hierarchical": "330m",
    "fusion": "330m",
}
SEED = 42
ARCHIVE_NAME = "ll_hls4ml_hierarchy_fusion_scale200_results"
KERNEL_TYPES = [
    "2layer", "3layer", "conv1d", "conv2d",
    "dense_latency", "dense_resource", "rule4ml",
]


def assert_commit(value, name):
    assert re.fullmatch(r"[0-9a-fA-F]{40,64}", value), (
        f"{name} must be an immutable full commit hash; got {value!r}"
    )


assert_commit(REPO_REF, "REPO_REF")
assert_commit(TENSOR_REVISION, "TENSOR_REVISION")
assert set(ACTIVE_RUNS) <= {"hierarchical", "fusion"}
for run_name, budget in TRAIN_BUDGETS.items():
    assert re.fullmatch(r"[1-9][0-9]*[smhd]", budget), (run_name, budget)

GPU_COUNT = torch.cuda.device_count()
assert GPU_COUNT >= 1, "Enable a Kaggle GPU accelerator before running."
PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "float32"
print("PyTorch:", torch.__version__)
print("CUDA devices:", GPU_COUNT)
for index in range(GPU_COUNT):
    print(index, torch.cuda.get_device_name(index))
print("Precision:", PRECISION)
print("DDP world size:", GPU_COUNT if USE_DDP else 1)
print("Active runs:", ACTIVE_RUNS)
print("Training budgets:", TRAIN_BUDGETS)


## Install dependencies and checkout immutable training code

In [ ]:
# Kaggle already supplies CUDA-enabled PyTorch.
%pip install -q torch-geometric "huggingface_hub[hf_xet]>=0.32" pyyaml pandas


In [ ]:
if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags"],
        check=True,
    )
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert commit == REPO_REF, (commit, REPO_REF)

sys.path.insert(0, str(REPO_DIR / "src"))
from ll_hls4ml.data.tensorize import EMBED_SIZE
from ll_hls4ml.io.schema import (
    BLOCK_FEATURE_SIZE,
    FUNCTION_FEATURE_SIZE,
    PRAGMA_FEATURE_SIZE,
)
from ll_hls4ml.models.registry import list_models

assert {"hierarchical", "hierarchical_high_level_fusion"} <= set(list_models())
HISTORICAL_SPLIT_MANIFEST_PATH = (
    REPO_DIR
    / "artifacts/results/ll_hls4ml_gatv2_scaling_results"
    / "kaggle_v2_gatv2_1head_scale200_seed42/split_manifest.json"
)
assert HISTORICAL_SPLIT_MANIFEST_PATH.is_file()
historical_split_manifest = json.loads(HISTORICAL_SPLIT_MANIFEST_PATH.read_text())
print("ll-hls4ml commit:", commit)
print("Historical split sizes:", {
    key: len(value) for key, value in historical_split_manifest.items()
})


## Build the expanded official split and download only its tensors

In [ ]:
from threading import Event, Thread
import huggingface_hub
from huggingface_hub import hf_hub_download, login, snapshot_download
from huggingface_hub.utils import enable_progress_bars

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
enable_progress_bars()
print("huggingface_hub:", huggingface_hub.__version__)

HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
index_path = Path(hf_hub_download(
    repo_id=TENSOR_REPO_ID, filename="labels.json", repo_type="dataset",
    revision=TENSOR_REVISION, token=hf_token, cache_dir=str(HF_CACHE_DIR),
))
tensor_index = json.loads(index_path.read_text())
metadata = tensor_index.get("metadata", {})

def archive_number(path):
    return int(Path(path).parts[1].removeprefix("archive_"))

def unique_by_graph_id(paths):
    selected = []
    seen = set()
    for path in sorted(paths):
        graph_id = Path(path).stem
        if graph_id not in seen:
            selected.append(path)
            seen.add(graph_id)
    return selected

main_paths = unique_by_graph_id(
    path for path in tensor_index["labels"]
    if Path(path).parts[0] in KERNEL_TYPES and archive_number(path) <= 8
)
exemplar_paths = unique_by_graph_id(
    path for path in tensor_index["labels"]
    if Path(path).parts[0] == "exemplar" and archive_number(path) <= 8
)
split_manifest = {name: [] for name in ("train", "validation", "test")}
for path in main_paths:
    split = str(metadata[path].get("dataset_split", "")).lower()
    split = "validation" if split in {"val", "validation"} else split
    assert split in split_manifest, f"Missing official split for {path}"
    split_manifest[split].append({
        "kernel_family": Path(path).parts[0], "tensor_path": path
    })
split_manifest["exemplar"] = [
    {"kernel_family": "exemplar", "tensor_path": path}
    for path in exemplar_paths
]
expected_sizes = {"train": 2742, "validation": 569, "test": 593, "exemplar": 799}
actual_sizes = {name: len(rows) for name, rows in split_manifest.items()}
assert actual_sizes == expected_sizes, (actual_sizes, expected_sizes)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_MANIFEST_PATH = CONFIG_DIR / "official_archives1_8_manifest.json"
SPLIT_MANIFEST_PATH.write_text(json.dumps(split_manifest, indent=2))
tensor_paths = sorted({
    row["tensor_path"] for rows in split_manifest.values() for row in rows
})
download_patterns = [
    f"{family}/archive_{archive}/*.pt"
    for family in (*KERNEL_TYPES, "exemplar")
    for archive in range(1, 9)
]
download_file_count = sum(
    1 for path in tensor_index["labels"]
    if Path(path).parts[0] in {*KERNEL_TYPES, "exemplar"}
    and archive_number(path) <= 8
)

def directory_size(path):
    total = 0
    if not path.exists():
        return total
    for item in path.rglob("*"):
        try:
            if item.is_file():
                total += item.stat().st_size
        except FileNotFoundError:
            pass  # A concurrent download may rename a temporary file.
    return total

def download_heartbeat(stop_event):
    previous = None
    last_change = time.monotonic()
    while not stop_event.wait(DOWNLOAD_HEARTBEAT_SECONDS):
        completed = len(list(TENSOR_DOWNLOAD_DIR.rglob("*.pt")))
        visible_bytes = (
            directory_size(TENSOR_DOWNLOAD_DIR) + directory_size(HF_CACHE_DIR)
        )
        state = (completed, visible_bytes)
        now = time.monotonic()
        if state != previous:
            last_change = now
            previous = state
        quiet_minutes = (now - last_change) / 60
        warning = (
            " WARNING: no visible disk progress"
            if quiet_minutes >= DOWNLOAD_STALL_WARNING_MINUTES else ""
        )
        print(
            f"[download heartbeat] {completed}/{download_file_count} tensor files; "
            f"{visible_bytes / 2**30:.2f} GiB visible; "
            f"unchanged {quiet_minutes:.1f} min.{warning}",
            flush=True,
        )

TENSOR_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
print(
    f"Downloading {download_file_count} tensors (about 16.84 GiB) from "
    f"{TENSOR_REPO_ID} at {TENSOR_REVISION}.",
    flush=True,
)
print("A stdout heartbeat will appear once per minute.", flush=True)
heartbeat_stop = Event()
heartbeat_thread = Thread(
    target=download_heartbeat, args=(heartbeat_stop,), daemon=True
)
heartbeat_thread.start()
try:
    TENSOR_DIR = Path(snapshot_download(
        repo_id=TENSOR_REPO_ID,
        repo_type="dataset",
        revision=TENSOR_REVISION,
        token=hf_token,
        cache_dir=str(HF_CACHE_DIR),
        local_dir=str(TENSOR_DOWNLOAD_DIR),
        allow_patterns=[*download_patterns, "labels.json", "vocab.json"],
    ))
finally:
    heartbeat_stop.set()
    heartbeat_thread.join()
print("Download call completed.", flush=True)
VOCAB_PATH = TENSOR_DIR / "vocab.json"
assert VOCAB_PATH.is_file()
print("Expanded split sizes:", actual_sizes)
print("Tensor snapshot:", TENSOR_DIR)


## Validate hierarchy tensors, cohort boundaries, and fusion alignment

In [ ]:
missing_tensors = [path for path in tensor_paths if not (TENSOR_DIR / path).is_file()]
assert not missing_tensors, f"Missing tensors; first: {missing_tensors[:5]}"

sample = torch.load(TENSOR_DIR / tensor_paths[0], map_location="cpu", weights_only=False)
assert "function" in sample.node_types and sample["function"].num_nodes > 0
assert sample["function"].x.shape[1] == FUNCTION_FEATURE_SIZE
assert sample["block"].x.shape[1] == BLOCK_FEATURE_SIZE
assert sample["pragma"].x.shape[1] == PRAGMA_FEATURE_SIZE
for node_type in ("variable", "constant"):
    assert sample[node_type].x.shape[1] == EMBED_SIZE
assert getattr(sample, "hierarchy_schema_version", None) == 2
for key in ("call_depth", "is_root", "is_entry", "is_reachable"):
    assert hasattr(sample["function"], key), f"Function hierarchy is missing {key}"
for node_type in ("instruction", "block"):
    assert hasattr(sample[node_type], "call_depth")

def archive_set(split):
    return {Path(row["tensor_path"]).parts[1] for row in split_manifest[split]}

for split in ("train", "validation", "test", "exemplar"):
    assert archive_set(split) == {f"archive_{index}" for index in range(1, 9)}
group_splits = {}
for split in ("train", "validation", "test"):
    for row in split_manifest[split]:
        path = row["tensor_path"]
        group_id = str(metadata[path].get("group_id") or Path(path).stem)
        key = (row["kernel_family"], group_id)
        previous = group_splits.setdefault(key, split)
        assert previous == split, f"Group crosses splits: {key}"

def resolve_high_level_cache():
    if HIGH_LEVEL_CACHE_SOURCE is not None:
        path = Path(HIGH_LEVEL_CACHE_SOURCE)
        assert path.is_file(), f"High-level cache not found: {path}"
        return path
    input_root = Path("/kaggle/input")
    matches = sorted(input_root.rglob("wa_high_level_archives1_8.pt"))
    if len(matches) != 1:
        raise FileNotFoundError(
            "Attach exactly one wa_high_level_archives1_8.pt Kaggle input or set "
            f"HIGH_LEVEL_CACHE_SOURCE explicitly; found {matches}"
        )
    return matches[0]

HIGH_LEVEL_CACHE_PATH = resolve_high_level_cache()
high_level_cache = torch.load(HIGH_LEVEL_CACHE_PATH, map_location="cpu", weights_only=False)
cache_paths = set(high_level_cache["samples"])
missing_cache = sorted(set(tensor_paths) - cache_paths)
assert not missing_cache, (
    f"The high-level cache does not cover the 200% manifest; first missing: "
    f"{missing_cache[:5]}"
)
print("Validated hierarchy tensor schema and exact cohort boundaries.")
print("High-level cache:", HIGH_LEVEL_CACHE_PATH)
print("High-level cached samples:", len(cache_paths))


## Matched configurations and cross-session resume support

In [ ]:
common = {
    "tensor_dir": str(TENSOR_DIR),
    "tensor_source_revision": TENSOR_REVISION,
    "vocab_path": str(VOCAB_PATH),
    "split_manifest_path": str(SPLIT_MANIFEST_PATH),
    "require_complete_split_manifest": True,
    "results_dir": str(RESULTS_DIR),
    "kernel_types": KERNEL_TYPES,
    "scale_percent": 200,
    "distributed_world_size": GPU_COUNT if USE_DDP else 1,
    "seed": SEED,
    "family_balanced_sampling": False,
    "batch_size": 8,
    "num_workers": 2,
    "worker_tmpdir": "/tmp",
    "pin_memory": True,
    "prefetch_factor": 2,
    "thread_prefetch": False,
    "precision": PRECISION,
    "epochs": 400,
    "patience": 30,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "hidden_dim": 64,
    "num_layers": 3,
    "dropout": 0.15,
    "use_global_features": True,
    "use_context": True,
    "context_mode": "core",
    "split_heads": True,
    "hurdle_heads": True,
    "hurdle_prediction_mode": "threshold",
    "loss": "log_huber_hurdle",
    "log_huber_delta": 0.35,
    "hurdle_classification_weight": 0.25,
    "early_stopping_metric": "smape",
    "verbose": 2,
}
experiments = {
    "hierarchical": "hierarchical_scale200_seed42",
    "fusion": "hierarchical_high_level_fusion_scale200_seed42",
}
run_configs = {
    "hierarchical": {**common, "model": "hierarchical"},
    "fusion": {
        **common,
        "model": "hierarchical_high_level_fusion",
        "heads": 1,
        "high_level_encoder": "gatv2",
        "high_level_cache": str(HIGH_LEVEL_CACHE_PATH),
    },
}
for name, config in run_configs.items():
    experiment = experiments[name]
    config["experiment_name"] = experiment
    config["checkpoint_dir"] = str(RESULTS_DIR / experiment / "checkpoints")

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
_cached_previous_root = None

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

SPLIT_MANIFEST_SHA256 = file_sha256(SPLIT_MANIFEST_PATH)
HIGH_LEVEL_CACHE_SHA256 = file_sha256(HIGH_LEVEL_CACHE_PATH)

def previous_search_root():
    global _cached_previous_root
    if PREVIOUS_RESULTS_ROOT is None:
        return None
    if _cached_previous_root is not None:
        return _cached_previous_root
    root = Path(PREVIOUS_RESULTS_ROOT)
    archives = sorted(root.rglob(f"{ARCHIVE_NAME}.zip"))
    if archives:
        assert len(archives) == 1, f"Multiple previous archives: {archives}"
        extracted = WORK / "previous_hierarchy_fusion_scale200_results"
        if not extracted.is_dir():
            shutil.unpack_archive(archives[0], extracted)
        root = extracted
    _cached_previous_root = root
    return root

def resume_signature(name, config):
    keys = [
        "model", "scale_percent", "seed", "batch_size", "precision",
        "distributed_world_size",
        "epochs", "patience", "learning_rate", "weight_decay",
        "hidden_dim", "num_layers", "heads", "dropout",
        "high_level_encoder", "use_global_features", "use_context",
        "context_mode", "split_heads", "hurdle_heads", "loss",
    ]
    signature = {key: config.get(key) for key in keys}
    signature.update({
        "repo_ref": REPO_REF,
        "tensor_source_revision": TENSOR_REVISION,
        "split_manifest_sha256": SPLIT_MANIFEST_SHA256,
    })
    if name == "fusion":
        signature["high_level_cache_sha256"] = HIGH_LEVEL_CACHE_SHA256
    return signature

def find_previous_run(experiment):
    root = previous_search_root()
    if root is None:
        return None
    matches = sorted(
        path.parent for path in root.rglob("notebook_resume_signature.json")
        if path.parent.name == experiment
    )
    assert len(matches) <= 1, f"Multiple previous runs for {experiment}: {matches}"
    return matches[0] if matches else None

def prepare_run(name):
    config = run_configs[name]
    experiment = config["experiment_name"]
    run_dir = RESULTS_DIR / experiment
    expected = resume_signature(name, config)
    previous = find_previous_run(experiment)
    if not run_dir.exists() and previous is not None:
        prior = json.loads((previous / "notebook_resume_signature.json").read_text())
        assert prior == expected, f"Refusing incompatible resume for {experiment}"
        shutil.copytree(previous, run_dir)
        print("Imported previous run:", previous)
    run_dir.mkdir(parents=True, exist_ok=True)
    signature_path = run_dir / "notebook_resume_signature.json"
    if signature_path.is_file():
        assert json.loads(signature_path.read_text()) == expected, (
            f"Local provenance changed for {experiment}"
        )
    signature_path.write_text(json.dumps(expected, indent=2))
    payload = dict(config)
    backup = Path(config["checkpoint_dir"]) / f"{experiment}_backup.pt"
    if backup.is_file():
        payload["resume_checkpoint_path"] = str(backup)
        print("Will resume", experiment, "from", backup)
    config_path = CONFIG_DIR / f"{experiment}.json"
    config_path.write_text(json.dumps(payload, indent=2))
    return config_path

for name in run_configs:
    prepare_run(name)


## Time-bounded training and incremental packaging

In [ ]:
import shlex

TRAIN_SCRIPT = REPO_DIR / "scripts/train.py"
run_environment = os.environ.copy()
run_environment["PYTHONPATH"] = str(REPO_DIR / "src")
run_environment["LL_HLS4ML_TQDM"] = "0"
run_environment["MPLCONFIGDIR"] = "/kaggle/working/matplotlib"

def base_training_command(config_path):
    if USE_DDP and GPU_COUNT > 1:
        return [
            sys.executable, "-m", "torch.distributed.run", "--standalone",
            f"--nproc_per_node={GPU_COUNT}", str(TRAIN_SCRIPT),
            "--config", str(config_path),
        ]
    return [sys.executable, str(TRAIN_SCRIPT), "--config", str(config_path)]

def run_and_stream(command, log_path):
    print("$", shlex.join(command), flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", buffering=1) as log:
        process = subprocess.Popen(
            command, cwd=REPO_DIR, env=run_environment,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        return process.wait()

def package_results():
    archive = Path(shutil.make_archive(
        str(WORK / ARCHIVE_NAME), "zip", root_dir=RESULTS_DIR
    ))
    print("Updated result archive:", archive)
    return archive

def run_experiment(name):
    if name not in ACTIVE_RUNS:
        print("Skipping", name, "by ACTIVE_RUNS configuration.")
        return
    config = run_configs[name]
    config_path = prepare_run(name)
    experiment = config["experiment_name"]
    run_dir = RESULTS_DIR / experiment
    summary_path = run_dir / "summary.json"
    if summary_path.is_file():
        summary = json.loads(summary_path.read_text())
        timeout_evaluation = summary.get("resolved_config", {}).get(
            "evaluation_checkpoint_path"
        )
        if timeout_evaluation is None:
            print(experiment, "already completed normally; skipping.")
            package_results()
            return
        print(experiment, "has a timeout evaluation; resuming training.")
    command = [
        "timeout", "--signal=INT", "--kill-after=5m", TRAIN_BUDGETS[name],
        *base_training_command(config_path),
    ]
    try:
        started = time.time()
        return_code = run_and_stream(command, run_dir / "training.log")
        print(experiment, "return code:", return_code)
        print(experiment, "wall seconds:", round(time.time() - started, 1))
        if return_code == 0 and summary_path.is_file():
            print("Training and evaluation completed normally.")
            return
        best_checkpoint = (
            Path(config["checkpoint_dir"]) / f"{experiment}_checkpoint.pt"
        )
        assert best_checkpoint.is_file(), (
            "The time limit expired before a best checkpoint existed. "
            "Use a budget long enough to complete at least one epoch."
        )
        evaluation_command = [
            sys.executable, str(TRAIN_SCRIPT), "--config", str(config_path),
            "--evaluate-checkpoint", str(best_checkpoint),
        ]
        evaluation_code = run_and_stream(
            evaluation_command, run_dir / "evaluation.log"
        )
        assert evaluation_code == 0, f"Evaluation failed: {evaluation_code}"
    finally:
        package_results()


## Run hierarchy-only at 200%

Run this cell in every session. It skips a normally completed run and resumes
a time-capped run when its optimizer backup is present.

In [ ]:
run_experiment("hierarchical")


## Run late fusion at 200%

This uses exactly the same split and CDFG model settings, adding only the
aligned high-level branch and learned late-fusion head.

In [ ]:
run_experiment("fusion")


## Compare available results and download the resume archive

In [ ]:
import numpy as np
import pandas as pd

TARGET_NAMES = ("lut", "ff", "dsp", "bram", "cycles_max", "interval_max")
historical_paths = {
    split: {row["tensor_path"] for row in historical_split_manifest[split]}
    for split in ("test", "exemplar")
}

def prediction_metrics(frame):
    smapes = []
    r2s = []
    for target in TARGET_NAMES:
        truth = frame[f"target_{target}"].to_numpy(dtype=float)
        prediction = frame[f"prediction_{target}"].to_numpy(dtype=float)
        denominator = np.abs(truth) + np.abs(prediction) + 1.0
        smape = 200 * np.abs(prediction - truth) / denominator
        residual = np.square(truth - prediction).sum()
        total = np.square(truth - truth.mean()).sum()
        smapes.append(smape.mean())
        r2s.append(1 - residual / total if total > 0 else float("nan"))
    return float(np.mean(smapes)), float(np.nanmean(r2s))

rows = []
for name, experiment in experiments.items():
    run_dir = RESULTS_DIR / experiment
    metrics_path = run_dir / "metrics.csv"
    summary_path = run_dir / "summary.json"
    if not (metrics_path.is_file() and summary_path.is_file()):
        continue
    metrics = pd.read_csv(metrics_path)
    metrics = metrics[metrics["kernel_family"] == "all"]
    summary = json.loads(summary_path.read_text())
    for split in ("test", "exemplar"):
        selected = metrics[metrics["split"] == split]
        rows.append({
            "model": name,
            "split": split,
            "macro_smape": selected["smape"].mean(),
            "macro_r2": selected["r2"].mean(),
            "best_epoch": summary.get("best_epoch"),
            "best_validation_smape": summary.get("best_metric"),
            "training_samples": summary["sizes"]["train"],
            "time_capped": bool(
                summary.get("resolved_config", {}).get("evaluation_checkpoint_path")
            ),
        })
    predictions = pd.read_csv(run_dir / "predictions.csv")
    for split in ("test", "exemplar"):
        selected = predictions[
            (predictions["split"] == split)
            & predictions["tensor_path"].isin(historical_paths[split])
        ]
        assert len(selected) == len(historical_paths[split]), (
            name, split, len(selected), len(historical_paths[split])
        )
        macro_smape, macro_r2 = prediction_metrics(selected)
        rows.append({
            "model": name,
            "split": f"{split}_historical_archives1_4",
            "macro_smape": macro_smape,
            "macro_r2": macro_r2,
            "best_epoch": summary.get("best_epoch"),
            "best_validation_smape": summary.get("best_metric"),
            "training_samples": summary["sizes"]["train"],
            "time_capped": bool(
                summary.get("resolved_config", {}).get("evaluation_checkpoint_path")
            ),
        })
comparison = pd.DataFrame(rows)
display(comparison)
if not comparison.empty:
    comparison.to_csv(RESULTS_DIR / "hierarchy_fusion_scale200_summary.csv", index=False)
archive = package_results()
from IPython.display import FileLink
display(FileLink(str(archive)))


## Interpretation guardrails

- A time-capped result is provisional even though the best checkpoint is evaluated.
- Compare the two models only when both report 2,742 training samples and the same tensor revision.
- Treat the expanded archive-1–8 metrics as the primary benchmark; use the archive-1–4 slices only for historical comparison.
- This is one seed. Confirm the direction with additional seeds before treating a small difference as architectural.
- Keep the produced zip: it contains both periodic optimizer backups needed for the next Kaggle session.
